# Generate ground truth matrices from the given structure files

In [1]:
import numpy as np
import pandas as pd
import os

def get_num_vars(timeseries_file):
    """
    Get number of variables from time series file.
    
    Args:
        timeseries_file: Path to time series CSV file
    Returns:
        Number of columns in the file
    """
    df = pd.read_csv(timeseries_file, nrows=1)
    return len(df.columns)

def create_adjacency_matrices(input_file, num_nodes):
    """
    Create adjacency matrices from structure file.
    
    Args:
        input_file: Path to input CSV file containing structure information
        num_nodes: Number of nodes in the network
    Returns:
        Tuple of (adjacency matrices list, maximum lag)
    """
    df = pd.read_csv(input_file, names=['cause', 'effect', 'lag'])
    max_lag = df['lag'].max()
    
    # Initialize adjacency matrices
    B_matrices = [np.zeros((num_nodes, num_nodes)) for _ in range(max_lag + 1)]
    
    # Populate the adjacency matrices
    for _, row in df.iterrows():
        cause, effect, lag = row['cause'], row['effect'], row['lag']
        B_matrices[lag][effect, cause] = 1
        
    print(f"Processing {input_file}")
    print(f"Max lag: {max_lag}")
    print(f"B_matrices shape: {B_matrices[0].shape}")
    
    return B_matrices, max_lag

def save_ground_truth(B_matrices, filepath):
    """
    Save adjacency matrices to file.
    
    Args:
        B_matrices: List of adjacency matrices
        filepath: Output file path
    """
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    
    with open(filepath, 'w') as f:
        for i, B in enumerate(B_matrices):
            np.savetxt(f, B, delimiter=',', fmt='%.3f')
            if i < len(B_matrices) - 1:
                f.write('\n')

def process_files(file_triplets):
    """
    Process multiple sets of input files.
    
    Args:
        file_triplets: List of (structure_file, ground_truth_file, returns_file) tuples
    """
    for structure_file, gt_file, returns_file in file_triplets:
        try:
            # Get number of variables from time series file
            n_vars = get_num_vars(returns_file)
            print(f"Number of variables: {n_vars}")
            
            # Create and save adjacency matrices
            B_matrices, _ = create_adjacency_matrices(structure_file, n_vars)
            save_ground_truth(B_matrices, gt_file)
            
            print(f"Saved adjacency matrices to {gt_file}\n")
            
        except Exception as e:
            print(f"Error processing files:\n"
                  f"Structure: {structure_file}\n"
                  f"Returns: {returns_file}\n"
                  f"Error: {str(e)}\n")

In [3]:
# Create file triplets
file_triplets = [
    (f'structures/sim{i}_gt_processed.csv',
        f'ground_truths/sim{i}_gt_processed_adj.csv',
        f'returns/timeseries{i}.csv')
    for i in range(1, 29)  # Process datasets 1-28
]

# Process all file triplets
process_files(file_triplets)

print("All files processed successfully.")

Number of variables: 5
Processing structures/sim1_gt_processed.csv
Max lag: 1
B_matrices shape: (5, 5)
Saved adjacency matrices to ground_truths/sim1_gt_processed_adj.csv

Number of variables: 10
Processing structures/sim2_gt_processed.csv
Max lag: 1
B_matrices shape: (10, 10)
Saved adjacency matrices to ground_truths/sim2_gt_processed_adj.csv

Number of variables: 15
Processing structures/sim3_gt_processed.csv
Max lag: 1
B_matrices shape: (15, 15)
Saved adjacency matrices to ground_truths/sim3_gt_processed_adj.csv

Number of variables: 50
Processing structures/sim4_gt_processed.csv
Max lag: 1
B_matrices shape: (50, 50)
Saved adjacency matrices to ground_truths/sim4_gt_processed_adj.csv

Number of variables: 5
Processing structures/sim5_gt_processed.csv
Max lag: 1
B_matrices shape: (5, 5)
Saved adjacency matrices to ground_truths/sim5_gt_processed_adj.csv

Number of variables: 10
Processing structures/sim6_gt_processed.csv
Max lag: 1
B_matrices shape: (10, 10)
Saved adjacency matrices 